In [5]:
import pandas as pd
import json


In [6]:
orders = pd.read_csv("orders.csv")

print("Orders Dataset Preview:")
orders.head()
orders.info()


Orders Dataset Preview:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_id         10000 non-null  int64  
 1   user_id          10000 non-null  int64  
 2   restaurant_id    10000 non-null  int64  
 3   order_date       10000 non-null  object 
 4   total_amount     10000 non-null  float64
 5   restaurant_name  10000 non-null  object 
dtypes: float64(1), int64(3), object(2)
memory usage: 468.9+ KB


In [7]:
with open("users.json") as f:
    users_data = json.load(f)

users = pd.DataFrame(users_data)

print("Users Dataset Preview:")
users.head()
users.info()


Users Dataset Preview:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   user_id     3000 non-null   int64 
 1   name        3000 non-null   object
 2   city        3000 non-null   object
 3   membership  3000 non-null   object
dtypes: int64(1), object(3)
memory usage: 93.9+ KB


In [12]:
import re

print('Parsing restaurants SQL file...')
with open("restaurants.sql", 'r', encoding='utf-8') as f:
    txt = f.read()


pattern = r"INSERT INTO restaurants VALUES \((\d+), '([^']+)', '([^']+)', ([0-9.]+)\);"
matches = re.findall(pattern, txt)

if not matches:
    raise SystemExit('No INSERT rows found in restaurants file. Check format.')

restaurants = pd.DataFrame(matches, columns=['restaurant_id', 'restaurant_name', 'cuisine', 'rating'])

restaurants['restaurant_id'] = restaurants['restaurant_id'].astype(int)
restaurants['rating'] = restaurants['rating'].astype(float)

print(f'Found {len(restaurants)} restaurants')

print("Restaurants Dataset Preview:")
restaurants.head()
restaurants.info()

Parsing restaurants SQL file...
Found 500 restaurants
Restaurants Dataset Preview:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   restaurant_id    500 non-null    int64  
 1   restaurant_name  500 non-null    object 
 2   cuisine          500 non-null    object 
 3   rating           500 non-null    float64
dtypes: float64(1), int64(1), object(2)
memory usage: 15.8+ KB


In [13]:
orders_users = pd.merge(
    orders,
    users,
    on="user_id",
    how="left"
)

orders_users.head()


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name,name,city,membership
0,1,2508,450,18-02-2023,842.97,New Foods Chinese,User_2508,Hyderabad,Regular
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular


In [14]:
final_df = pd.merge(
    orders_users,
    restaurants,
    on="restaurant_id",
    how="left"
)

final_df.head()


,order_id,user_id,restaurant_id,order_date,total_amount,restaurant_name_x,name,city,membership,restaurant_name_y,cuisine,rating
0,1,2508,450,18-02-2023,842.97,New Foods Chinese,User_2508,Hyderabad,Regular,Restaurant_450,Mexican,3.2
1,2,2693,309,18-01-2023,546.68,Ruchi Curry House Multicuisine,User_2693,Pune,Regular,Restaurant_309,Indian,4.5
2,3,2084,107,15-07-2023,163.93,Spice Kitchen Punjabi,User_2084,Chennai,Gold,Restaurant_107,Mexican,4.0
3,4,319,224,04-10-2023,1155.97,Darbar Kitchen Non-Veg,User_319,Bangalore,Gold,Restaurant_224,Chinese,4.8
4,5,1064,293,25-12-2023,1321.91,Royal Eatery South Indian,User_1064,Pune,Regular,Restaurant_293,Italian,3.0


In [15]:
final_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   order_id           10000 non-null  int64  
 1   user_id            10000 non-null  int64  
 2   restaurant_id      10000 non-null  int64  
 3   order_date         10000 non-null  object 
 4   total_amount       10000 non-null  float64
 5   restaurant_name_x  10000 non-null  object 
 6   name               10000 non-null  object 
 7   city               10000 non-null  object 
 8   membership         10000 non-null  object 
 9   restaurant_name_y  10000 non-null  object 
 10  cuisine            10000 non-null  object 
 11  rating             10000 non-null  float64
dtypes: float64(2), int64(3), object(7)
memory usage: 937.6+ KB


In [16]:
final_df.isnull().sum()


,0
order_id,0
user_id,0
restaurant_id,0
order_date,0
total_amount,0
restaurant_name_x,0
name,0
city,0
membership,0
restaurant_name_y,0


In [17]:
final_df.to_csv("final_food_delivery_dataset.csv", index=False)

print("Final dataset saved successfully!")


Final dataset saved successfully!


In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [20]:
df = pd.read_csv("final_food_delivery_dataset.csv")
df.head()
df = df.drop_duplicates()


In [22]:
gold_df = df[df["membership"] == "Gold"]

revenue_by_city_gold = gold_df.groupby("city")["total_amount"].sum() \
                               .sort_values(ascending=False)

revenue_by_city_gold

,total_amount
city,
Chennai,1080909.79
Pune,1003012.32
Bangalore,994702.59
Hyderabad,896740.19


In [24]:
avg_order_by_cuisine = df.groupby("cuisine")["total_amount"].mean() \
                          .sort_values(ascending=False)

avg_order_by_cuisine

,total_amount
cuisine,
Mexican,808.021344
Italian,799.448578
Indian,798.466011
Chinese,798.389020


In [26]:
user_spend = df.groupby("user_id")["total_amount"].sum()

In [27]:
high_value_users = user_spend[user_spend > 1000]
len(high_value_users)


2544

In [28]:
bins = [0, 500, 1000, 2000, np.inf]
labels = ["< 500", "500 – 1000", "1000 – 2000", "> 2000"]

bucketed = pd.cut(user_spend, bins=bins, labels=labels)
bucketed.value_counts()


,count
total_amount,
> 2000,1845
1000 – 2000,699
500 – 1000,225
< 500,114


In [29]:
rating_bins = [3.0, 3.5, 4.0, 4.5, 5.0]
rating_labels = ["3.0-3.5", "3.6-4.0", "4.1-4.5", "4.6-5.0"]

df["rating_range"] = pd.cut(df["rating"], bins=rating_bins, labels=rating_labels)


In [31]:
df.groupby("rating_range")["total_amount"].sum() \
  .sort_values(ascending=False)
gold_df.groupby("city")["total_amount"].mean() \
       .sort_values(ascending=False)
df.groupby("cuisine")["restaurant_id"].nunique() \
  .sort_values()
df.groupby("cuisine")["total_amount"].sum() \
  .sort_values(ascending=False)
gold_percentage = (len(df[df["membership"]=="Gold"]) / len(df)) * 100
round(gold_percentage)

/tmp/ipython-input-445744728.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("rating_range")["total_amount"].sum() \


50

In [33]:
rest_stats = df.groupby("restaurant_name_y")["total_amount"] \
               .agg(["mean","count"])

rest_stats[rest_stats["count"] < 20] \
    .sort_values("mean", ascending=False) \
    .head(40)

,mean,count
restaurant_name_y,,
Restaurant_294,1040.222308,13
Restaurant_262,1029.473333,18
Restaurant_77,1029.180833,12
Restaurant_193,1026.306667,15
Restaurant_7,1002.140625,16
Restaurant_298,989.822000,15
Restaurant_56,989.467222,18
Restaurant_135,988.702222,18
Restaurant_343,986.026471,17


In [36]:
df.groupby(["membership","cuisine"])["total_amount"] \
  .sum() \
  .sort_values(ascending=False) \
  .head(10)

membership  cuisine
Regular     Mexican    1072943.30
            Italian    1018424.75
Gold        Mexican    1012559.79
            Italian    1005779.05
Regular     Indian      992100.27
Gold        Indian      979312.31
            Chinese     977713.74
Regular     Chinese     952790.91
Name: total_amount, dtype: float64

In [37]:
df["order_date"] = pd.to_datetime(df["order_date"], dayfirst=True)
df["quarter"] = df["order_date"].dt.quarter
df.groupby("quarter")["total_amount"].sum() \
  .sort_values(ascending=False)

,total_amount
quarter,
3,2037385.10
4,2018263.66
1,2010626.64
2,1945348.72


In [44]:
gold_orders_count = len(df[df["membership"] == "Gold"])
print("gold_orders_count", gold_orders_count)

gold_orders_count 4987


In [47]:
revenue_hyderabad = int(
    df[df["city"] == "Hyderabad"]["total_amount"].sum()
)
print("revenue_hyderabad", revenue_hyderabad)

revenue_hyderabad 1889366


In [46]:
distinct_users = df["user_id"].nunique()
print("distinct_users", distinct_users)


distinct_users 2883


In [49]:
avg_gold = round(
    df[df["membership"] == "Gold"]["total_amount"].mean(),
    2
)
print("avg_gold", avg_gold)

avg_gold 797.15


In [50]:
orders_rating_ge_45 = len(df[df["rating"] >= 4.5])
print("orders_rating_ge_45", orders_rating_ge_45)


orders_rating_ge_45 3374


In [52]:
top_city_gold = (
    df[df["membership"]=="Gold"]
      .groupby("city")["order_id"]
      .count()
      .sort_values(ascending=False)
      .idxmax()
)

print("top_city_gold", top_city_gold)


top_city_gold Chennai


In [53]:
orders_in_top_city_gold = (
    df[
        (df["membership"]=="Gold") &
        (df["city"]==top_city_gold)
    ]["order_id"].count()
)

print("orders_in_top_city_gold", orders_in_top_city_gold)


orders_in_top_city_gold 1337
